# ClearPath DQR + Cleaning Pipeline

**Data Source**: Docker MySQL `clearpath` (loaded by `database_build.ipynb`)

**Structure**: Executive Summary → Data Profiling → Quality Dimensions → Anomaly Detection → DQ Score → Cleaning Pipeline → Action Items → Appendix

**Target ML**: Busyness prediction (time series + user reports)

---
## Part 0: Configuration & DB Connection

In [ ]:
import sys
from pathlib import Path
from datetime import datetime

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# ── Import ALL shared modules (single location) ──
UTIL_DIR = Path.cwd().resolve().parent / 'shared'
sys.path.insert(0, str(UTIL_DIR))
from dqr_utils import get_conn, is_manhattan, gps_to_district, MANHATTAN_BOUNDS
from dqr_io import load_dqr_tables, export_dqr_artifacts, build_audit_report
from dqr_checks import (
    run_all_checks, compute_dq_scores, compute_total_score,
    check_completeness, check_accuracy, check_consistency,
    check_uniqueness, check_timeliness, check_validity,
    check_database_integrity, check_fk_orphans,
    EXPECTED_EVENT_TYPES, DQ_WEIGHTS, VALID_VENUE_TYPES,
)
from dqr_analysis import (
    build_all_profiles, build_record_analysis,
    detect_coordinate_anomalies, detect_gps_duplicates,
    build_action_items, assess_ml_usability,
)
from dqr_cleaning import clean_venues
from external_ingestion import fetch_traffic_hourly, clean_traffic, fetch_and_clean_weather

# ── Paths ──
OUTPUT_DIR = Path.cwd() / 'output'
OUTPUT_DIR.mkdir(exist_ok=True)

DQR_TABLES = [
    'venues', 'restroom_profiles', 'healthcare_profiles',
    'emergency_assets', 'pedestrian_ramps', 'venue_source_links',
    'busyness_scores', 'external_context_cache', 'user_reports'
]

# ── Connect ──
conn = get_conn()
with conn.cursor() as cur:
    cur.execute('SELECT COUNT(*) FROM venues')
    venue_count = cur.fetchone()[0]
print(f'MySQL OK — {venue_count} venues | Output: {OUTPUT_DIR} | {datetime.now():%Y-%m-%d %H:%M}')

---
## 1. Executive Summary

In [ ]:
data = load_dqr_tables(conn, DQR_TABLES)
venues_df = data.get('venues', pd.DataFrame())
total_rows = sum(len(df) for df in data.values() if not df.empty)
tables_loaded = sum(1 for df in data.values() if not df.empty)
vt_dist = venues_df['venue_type'].value_counts().to_dict() if not venues_df.empty else {}
completeness = 100 - venues_df.isnull().mean().mean() * 100 if not venues_df.empty else 0

print(f'={"="*58}')
print(f'  EXECUTIVE SUMMARY — ClearPath DQR')
print(f'  数据库: clearpath (Docker MySQL)')
print(f'  总行数: {total_rows:,}  |  表加载: {tables_loaded}/{len(DQR_TABLES)}')
print(f'  场馆: {len(venues_df):,}  |  类型: {vt_dist}')
print(f'  完整率: {completeness:.1f}%  |  日期: {datetime.now():%Y-%m-%d}')
print(f'={"="*58}')

---
## 2. Data Profiling

In [ ]:
all_profiles = build_all_profiles(data)
print(f'Column profiling: {len(all_profiles)} fields across {tables_loaded} tables')

# Healthcare feature profile
hp = data.get('healthcare_profiles', pd.DataFrame())
if not hp.empty:
    rows = []
    for c in hp.columns:
        s = hp[c]
        nn = int(s.notna().sum())
        vc = s.value_counts(dropna=True).head(5)
        rows.append({
            'feature': c, 'non_null': nn, 'cardinality': int(s.nunique(dropna=True)),
            'top5': ' | '.join(str(v) for v in vc.index),
            'freq_%': ' | '.join(f'{int(v)/nn*100:.1f}' if nn else '0' for v in vc.values),
        })
    display(pd.DataFrame(rows).sort_values(['cardinality', 'non_null'], ascending=[False, False]))

# User report event types
ur = data.get('user_reports', pd.DataFrame())
if not ur.empty and 'issue_type' in ur.columns:
    actual = set(ur['issue_type'].dropna().unique())
    print(f'Event types: {len(actual)} found, unknown={sorted(actual - EXPECTED_EVENT_TYPES)}, missing={sorted(EXPECTED_EVENT_TYPES - actual)}')
else:
    print(f'Event types (reference): {sorted(EXPECTED_EVENT_TYPES)}')

# Row-level stats
row_stats = []
for name, df in data.items():
    if df.empty: continue
    outliers = 0
    if 'latitude' in df.columns:
        valid = df.dropna(subset=['latitude', 'longitude'])
        outliers = (~valid.apply(lambda r: is_manhattan(float(r['latitude']), float(r['longitude'])), axis=1)).sum()
    row_stats.append({'table': name, 'total': len(df), 'null_rows': df.isnull().all(axis=1).sum(),
                      'dups': df.duplicated().sum(), 'coord_outliers': outliers})
print(pd.DataFrame(row_stats).to_string(index=False))

# Record quality scores
record_analysis = build_record_analysis(venues_df)
if not record_analysis.empty:
    qs = record_analysis['record_quality_score']
    print(f'Record quality: mean={qs.mean():.2f}, min={qs.min():.2f}, low(<0.5)={(qs<0.5).sum()}')

In [ ]:
# FK / Referential Integrity
fk = check_fk_orphans(venues_df, data)
print('FK checks:', 'OK' if not fk['issues'] else fk['issues'])
if not venues_df.empty and 'venue_type' in venues_df.columns:
    invalid = set(venues_df['venue_type'].unique()) - VALID_VENUE_TYPES
    print(f'venue_type ENUM: invalid={invalid or "none"}')

---
## 3. Data Quality Dimensions

In [ ]:
# Run ALL quality checks in one call
check_results = run_all_checks(data, venues_df)
coord_valid_mask = check_results.pop('coord_valid_mask', None)
event_types = check_results.pop('event_types', None)

for name in ['completeness', 'accuracy', 'consistency', 'uniqueness', 'timeliness', 'validity', 'integrity', 'fk_orphans']:
    r = check_results[name]
    status = 'PASS' if r['passed'] else 'FAIL'
    issues = f'  issues={r["issues"]}' if r['issues'] else ''
    print(f'  {name:<15} score={r["score"]:>6.1f}  {status}{issues}')

# Show completeness details
print('\nCompleteness detail:')
print(check_results['completeness']['_dataframe'].to_string(index=False))

---
## 4. Anomaly Detection

In [ ]:
anomaly_df = detect_coordinate_anomalies(data)
print(f'Anomalies: {len(anomaly_df)}')
if not anomaly_df.empty:
    print(anomaly_df['type'].value_counts().to_string())

gps_sources = {k: v for k, v in data.items() if not v.empty and 'latitude' in v.columns and 'longitude' in v.columns}
gps_duplicates_df = detect_gps_duplicates(gps_sources)
print(f'GPS duplicates (<30m): {len(gps_duplicates_df)} pairs')
if not gps_duplicates_df.empty:
    print(gps_duplicates_df.head(5).to_string(index=False))

---
## 5. DQ Score & Rating

In [ ]:
scores = compute_dq_scores(venues_df, data, anomaly_df, gps_duplicates_df, coord_valid_mask)
total_score, grade = compute_total_score(scores)

print(f'{"Dimension":<18} {"Weight":>7} {"Score":>7} {"Weighted":>8}')
print('-' * 42)
for d in DQ_WEIGHTS:
    print(f'{d:<18} {DQ_WEIGHTS[d]:>6.0%} {scores[d]:>7.1f} {scores[d]*DQ_WEIGHTS[d]:>8.1f}')
print('-' * 42)
print(f'{"TOTAL":<18} {"100%":>7} {total_score:>7.1f}')
print(f'Rating: {grade} ({total_score:.1f}/100)')

In [ ]:
# Radar chart
fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))
dims = list(scores.keys())
vals = [scores[d] for d in dims]
angles = np.linspace(0, 2*np.pi, len(dims), endpoint=False).tolist()
vals_plot = vals + [vals[0]]; angles += angles[:1]
ax.fill(angles, vals_plot, alpha=0.25, color='steelblue')
ax.plot(angles, vals_plot, 'o-', color='steelblue', linewidth=2)
ax.set_xticks(angles[:-1]); ax.set_xticklabels(dims, size=11)
ax.set_ylim(0, 100)
ax.set_title(f'DQ Score: {total_score:.1f}/100 ({grade})', size=14, pad=20)
plt.tight_layout(); plt.savefig(OUTPUT_DIR / 'dqr_dimension_scores.png', dpi=150); plt.show()

# Missing rate heatmap
vf = all_profiles[all_profiles['table'] == 'venues'].copy()
if not vf.empty:
    fig, ax = plt.subplots(figsize=(12, 2))
    sns.heatmap(vf[['column', 'missing_pct']].set_index('column').T,
                annot=True, fmt='.0f', cmap='YlOrRd', ax=ax, vmin=0, vmax=100,
                cbar_kws={'orientation': 'horizontal'})
    ax.set_title('Venues — Missing Rate (%)')
    plt.tight_layout(); plt.savefig(OUTPUT_DIR / 'dqr_missing_heatmap.png', dpi=150); plt.show()

---
## 6. Cleaning Pipeline & External Data

In [ ]:
# Clean venues
venues_clean = clean_venues(venues_df, coord_valid_mask,
                           record_analysis['record_quality_score'] if not record_analysis.empty else None)
print(f'venues_clean: {len(venues_clean)} records')

# Scatter plot
if not venues_clean.empty and 'latitude' in venues_clean.columns:
    fig, ax = plt.subplots(figsize=(10, 12))
    colors = {'restroom': '#3498db', 'healthcare': '#e74c3c', 'emergency_asset': '#f39c12',
              'clinic': '#9b59b6', 'pharmacy': '#2ecc71', 'hospital': '#e91e63',
              'dentist': '#00bcd4', 'laboratory': '#795548'}
    for vt in venues_clean['venue_type'].unique():
        sub = venues_clean[venues_clean['venue_type'] == vt]
        ax.scatter(sub['longitude'], sub['latitude'], c=colors.get(vt, '#95a5a6'),
                  label=vt, alpha=0.5, s=10, edgecolors='none')
    ax.set(xlabel='Longitude', ylabel='Latitude',
           title=f'Manhattan Venues ({len(venues_clean)} records, {venues_clean["venue_type"].nunique()} types)',
           xlim=(MANHATTAN_BOUNDS['lng_min'], MANHATTAN_BOUNDS['lng_max']),
           ylim=(MANHATTAN_BOUNDS['lat_min'], MANHATTAN_BOUNDS['lat_max']))
    ax.legend(fontsize=8, markerscale=3)
    plt.tight_layout(); plt.savefig(OUTPUT_DIR / 'dqr_venue_scatter.png', dpi=150); plt.show()

In [ ]:
# External data
try:
    traffic_clean = clean_traffic(fetch_traffic_hourly(year=2025))
except Exception as e:
    print(f'Traffic failed: {e}'); traffic_clean = pd.DataFrame()

weather_clean = fetch_and_clean_weather()

In [ ]:
# Export all artifacts to output/
export_dqr_artifacts(OUTPUT_DIR, venues_clean=venues_clean, traffic_clean=traffic_clean,
                     weather_clean=weather_clean, field_summary=all_profiles,
                     record_analysis=record_analysis, anomalies=anomaly_df,
                     gps_duplicates=gps_duplicates_df)

---
## 7. Action Items & Appendix

In [ ]:
actions_df = build_action_items(venues_df, data, scores)
print('=== Action Items ===')
print(actions_df.to_string(index=False))

# Audit log
audit = build_audit_report(total_score=total_score, grade=grade, tables_loaded=tables_loaded,
                           total_rows=total_rows, venues_df=venues_df, venues_clean=venues_clean,
                           anomaly_df=anomaly_df, gps_duplicates_df=gps_duplicates_df,
                           actions_df=actions_df, output_dir=OUTPUT_DIR)

# ML Usability
ml = assess_ml_usability(venues_clean, traffic_clean, weather_clean, scores, grade)
print(f'\n{"="*50}')
print(f'  ML USABILITY: {ml["venues_count"]} venues, {ml["coord_complete_pct"]:.0f}% coords, '
      f'{ml["district_count"]}/4 districts, DQ={ml["dq_score"]:.1f}/100')
print(f'{"="*50}')

conn.close()
print('MySQL connection closed.')